<h1><center>Part 2: Fine-Tuning Diffusion Models for Domain-Specific Image Generation</center></h1>

Fine-tune a pretrained text-to-image diffusion model on a domain-specific dataset and evaluate the generated images using quantitative and qualitative metrics. Your goal is to:
1. Fine-tune a pretrained diffusion model (e.g., SD 1.5, SDXL, Flux, Qwen) on a captioned image dataset.
2. Generate images from structured text prompts.
3. Evaluate image quality, diversity, and text alignment using standard generative metrics.
4. Compare fine-tuning strategies and analyze trade-offs between compute cost and performance.

<b> Datasets: Naruto BLIP Captions: https://huggingface.co/datasets/lambdalabs/naruto-blip-captions </b>

In [1]:
import os
import gc
import math
import time
import json
import random
from pathlib import Path
from dataclasses import dataclass, asdict

import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import make_grid, save_image

from transformers import CLIPProcessor, CLIPModel
from diffusers import (
    StableDiffusionPipeline,
    DDPMScheduler,
    UNet2DConditionModel,
    AutoencoderKL,
)
from diffusers.optimization import get_cosine_schedule_with_warmup

from peft import LoraConfig, get_peft_model

# Reproducibility setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device setup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"DEVICE: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


/home/sandy/miniconda3/envs/DATA266/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sandy/miniconda3/envs/DATA266/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


DEVICE: cpu
